# Модуль 3 · Сбор и разметка данных
**Практикум.** Собираем данные с сайта, проверяем снимки на входе и меряем, насколько согласны разметчики.

**Как работать.** Меню **«Среда выполнения → Выполнить все»** (Runtime → Run all) — и читайте сверху вниз.
В ячейках с полями справа меняйте значения и запускайте ячейку заново (▶ слева или Ctrl+Enter). Сломать ничего
нельзя: **«Среда выполнения → Отключиться и удалить среду»** вернёт всё как было. Код можно не читать — но если
интересно, он открыт и короткий.

Снимки листьев — открытый набор **PlantVillage** (Mohanty, Hughes, Salathé, 2016), лицензия CC BY-SA 3.0, репозиторий [spMohanty/PlantVillage-Dataset](https://github.com/spMohanty/PlantVillage-Dataset).

## 1. Задание 1. Новости → CSV
Программа парсинга — четыре шага: **загрузить страницу → найти повторяющийся блок → достать из него поля →
сохранить**. Сначала потренируемся на учебной странице, встроенной в ноутбук: она работает без интернета и не
меняется.

In [ ]:
#@title Учебная страница: разбор по CSS-селекторам
import re, time, warnings
import pandas as pd, requests, matplotlib.pyplot as plt
from bs4 import BeautifulSoup
from collections import Counter
warnings.filterwarnings('ignore')

HTML = """    <html><body><div class="news-list">
<div class="news-card"><a class="news-card__title" href="/news/101">Студенты собрали дрон для мониторинга полей</a><span class="news-card__date">03.06.2026</span><span class="news-card__tag">Наука</span></div>
<div class="news-card"><a class="news-card__title" href="/news/102">Открыт приём на летнюю школу по данным</a><span class="news-card__date">17.06.2026</span><span class="news-card__tag">Учёба</span></div>
<div class="news-card"><a class="news-card__title" href="/news/103">Команда вышла в финал хакатона</a><span class="news-card__date">28.06.2026</span><span class="news-card__tag">Наука</span></div>
<div class="news-card"><a class="news-card__title" href="/news/104">Лаборатория получила новый стенд</a><span class="news-card__date">09.07.2026</span><span class="news-card__tag">Наука</span></div>
<div class="news-card"><a class="news-card__title" href="/news/105">Расписание осеннего семестра</a><span class="news-card__date">21.07.2026</span><span class="news-card__tag">Учёба</span></div>
<div class="news-card"><a class="news-card__title" href="/news/106">Итоги приёмной кампании</a><span class="news-card__date">05.08.2026</span><span class="news-card__tag">Учёба</span></div>
<div class="news-card"><a class="news-card__title" href="/news/107">Статья о разметке снимков принята в журнал</a><span class="news-card__date">14.08.2026</span><span class="news-card__tag">Наука</span></div>
<div class="news-card"><a class="news-card__title" href="/news/108">Спортивный праздник первокурсников</a><span class="news-card__date">01.09.2026</span><span class="news-card__tag">Жизнь</span></div>
<div class="news-card"><a class="news-card__title" href="/news/109">Семинар по компьютерному зрению</a><span class="news-card__date">11.09.2026</span><span class="news-card__tag">Наука</span></div>
<div class="news-card"><a class="news-card__title" href="/news/110">Грант на исследование датчиков почвы</a><span class="news-card__date">24.09.2026</span><span class="news-card__tag">Наука</span></div>
<div class="news-card"><a class="news-card__title" href="/news/111">День открытых дверей</a><span class="news-card__date">02.10.2026</span><span class="news-card__tag">Жизнь</span></div>
<div class="news-card"><a class="news-card__title" href="/news/112">Конференция молодых учёных</a><span class="news-card__date">15.10.2026</span><span class="news-card__tag">Наука</span></div>
</div></body></html>"""

item_selector = 'div.news-card'  #@param {type:"string"}
title_selector = 'a.news-card__title'  #@param {type:"string"}
date_selector = 'span.news-card__date'  #@param {type:"string"}

MONTHS = {m: i for i, m in enumerate(['января', 'февраля', 'марта', 'апреля', 'мая', 'июня', 'июля', 'августа',
                                      'сентября', 'октября', 'ноября', 'декабря'], 1)}
def parse_date(s):
    """Понимает «18.10.2026» и «18 октября 2026»."""
    s = (s or '').strip().lower()
    m = re.search(r'(\d{1,2})\.(\d{1,2})\.(\d{4})', s)
    if m:
        return pd.Timestamp(int(m[3]), int(m[2]), int(m[1]))
    m = re.search(r'(\d{1,2})\s+([а-яё]+)\s+(\d{4})', s)
    if m and m[2] in MONTHS:
        return pd.Timestamp(int(m[3]), MONTHS[m[2]], int(m[1]))
    return pd.NaT

def extract(html, base_url=''):
    soup = BeautifulSoup(html, 'html.parser')
    rows = []
    for block in soup.select(item_selector):
        t, d = block.select_one(title_selector), block.select_one(date_selector)
        link = t.get('href') if t is not None and t.has_attr('href') else (block.find('a') or {}).get('href')
        rows.append({'заголовок': t.get_text(strip=True) if t else None,
                     'дата': parse_date(d.get_text(strip=True)) if d else pd.NaT,
                     'ссылка': requests.compat.urljoin(base_url, link) if link else None})
    return pd.DataFrame(rows)

df = extract(HTML, 'https://example.org/')
df.to_csv('news_offline.csv', index=False)
print('Найдено новостей:', len(df), '→ сохранено в news_offline.csv (значок папки слева)')
df

**Что сделать.** Сотрите `div.` в поле `item_selector` — изменится ли результат? Впишите несуществующий класс —
что вернёт программа? Так выглядит самая частая поломка парсера: сайт переименовал класс.

### Теперь настоящий сайт
1. Откройте страницу новостей в браузере (QR в презентации), нажмите правой кнопкой на заголовок новости →
**«Просмотреть код»** (Inspect).
2. Найдите блок, который **повторяется** для каждой новости, и его класс. Внутри — заголовок и дату.
3. Впишите селекторы ниже. Если не знаете, с чего начать, — оставьте `item_selector` пустым: программа покажет
классы, которые повторяются на странице чаще всего.

In [ ]:
#@title Парсим живую страницу
url = 'https://mai.ru/press/news/'  #@param {type:"string"}
live_item = ''  #@param {type:"string"}
live_title = ''  #@param {type:"string"}
live_date = ''  #@param {type:"string"}
HEADERS = {'User-Agent': 'Mozilla/5.0 (учебный парсер курса Прикладное ML)'}
try:
    r = requests.get(url, headers=HEADERS, timeout=30)
    print('Код ответа:', r.status_code, '(200 — всё хорошо; 403 — сайт не пускает; 429 — слишком часто)')
    if not live_item:
        soup = BeautifulSoup(r.text, 'html.parser')
        cnt = Counter(t.name + '.' + '.'.join(t.get('class')) for t in soup.find_all(class_=True))
        print('\nПовторяются чаще всего (кандидаты в item_selector):')
        for k, v in cnt.most_common(40):
            if v >= 5:
                print(f'  {v:>3} × {k}')
    else:
        item_selector, title_selector, date_selector = live_item, live_title, live_date
        live = extract(r.text, url)
        live.to_csv('news_live.csv', index=False)
        print('Найдено новостей:', len(live), '→ news_live.csv')
        display(live.head(20))
except Exception as e:
    print('Страница не загрузилась:', e)

## 2. Задание 2. Одна категория за всё время → график по месяцам
Откройте вторую страницу новостей (или страницу категории) и посмотрите, как меняется адрес: обычно появляется
номер страницы, например `?PAGEN_1=2` или `/page/2/`. Впишите шаблон адреса с `{page}` вместо номера.
Программа идёт по страницам с паузой в секунду и останавливается, когда новые новости перестают появляться.
Пока шаблон пуст, график строится по учебной странице.

In [ ]:
#@title Обход страниц и график
page_template = ''  #@param {type:"string"}
max_pages = 20  #@param {type:"integer"}
if page_template:
    frames, seen = [], set()
    for page in range(1, max_pages + 1):
        u = page_template.format(page=page)
        r = requests.get(u, headers=HEADERS, timeout=30)
        if r.status_code != 200:
            print('стоп: код', r.status_code, 'на', u); break
        part = extract(r.text, u)
        part = part[~part['ссылка'].isin(seen)]
        if part.empty:
            print('стоп: новых новостей нет на странице', page); break
        seen.update(part['ссылка']); frames.append(part)
        print(f'страница {page}: +{len(part)}')
        time.sleep(1)                          # вежливость к сайту: не чаще раза в секунду
    data = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame(columns=['заголовок', 'дата', 'ссылка'])
    data.to_csv('news_category.csv', index=False)
else:
    data = df
    print('Шаблон пуст — строим по учебной странице')
by_month = data.dropna(subset=['дата']).set_index('дата').resample('MS').size()
ax = by_month.plot(kind='bar', figsize=(9, 3.5), color='#48A1FA')
ax.set_xticklabels([d.strftime('%m.%Y') for d in by_month.index], rotation=0)
ax.set_ylabel('новостей'); ax.set_xlabel(''); plt.show()

**Правила вежливого парсинга:** пауза между запросами, свой User-Agent, `robots.txt` и правила сайта, только
открытые данные, никаких персональных данных без основания (пять правовых вопросов — в презентации).

## 3. Контроль снимков на входе: резкость и экспозиция
**Резкость** — разброс оператора Лапласа: у резкого снимка много перепадов яркости. **Экспозиция** — доля
почти чёрных и почти белых пикселей.

In [ ]:
#@title Подготовка: библиотеки и доступ к снимкам PlantVillage
import io, os, subprocess, urllib.parse, urllib.request, warnings
from concurrent.futures import ThreadPoolExecutor
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
warnings.filterwarnings('ignore')

BASE = 'https://raw.githubusercontent.com/spMohanty/PlantVillage-Dataset/master/'
SAMPLES = {
    'apple_rust': 'raw/color/Apple___Cedar_apple_rust/46597071-9826-4f76-b853-083b76c5ef63___FREC_C.Rust 3666.JPG',
    'corn_gls': 'raw/color/Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot/8858dd29-eac0-443a-8213-425bbe223f7e___RS_GLSp 7334.JPG',
    'corn_healthy': 'raw/color/Corn_(maize)___healthy/3ac3f13d-352c-4390-bf06-94b5334cbfd8___R.S_HL 8206 copy 2.jpg',
    'grape_brot': 'raw/color/Grape___Black_rot/83b866ce-0785-4312-8e43-1ae8aa59d3bd___FAM_B.Rot 3518.JPG',
    'peach_bact': 'raw/color/Peach___Bacterial_spot/1df78012-4b87-40f4-bfa1-b868465c0b4d___Rut._Bact.S 1263.JPG',
    'soy_healthy': 'raw/color/Soybean___healthy/8f1ff646-db74-4924-abe8-373894711d26___RS_HL 5701.JPG',
    'tom_bact': 'raw/color/Tomato___Bacterial_spot/47798da0-ff59-497e-a2bb-e169c9138a7e___GCREC_Bact.Sp 3120.JPG',
    'tom_healthy': 'raw/color/Tomato___healthy/18fa8f7f-ef94-4a3b-9c79-5f4362323134___RS_HL 9977.JPG',
    'tom_late': 'raw/color/Tomato___Late_blight/6578c88a-fd14-450b-8e75-c94571decf83___GHLB2 Leaf 8849.JPG',
    'tom_mites': 'raw/color/Tomato___Spider_mites Two-spotted_spider_mite/2c2f7f98-8c5b-4981-bb74-f827c5f963fe___Com.G_SpM_FL 8734.JPG',
    'tom_mosaic': 'raw/color/Tomato___Tomato_mosaic_virus/6ebb483a-d1b6-4d7d-aad1-e717c2a36167___PSU_CG 2286.JPG',
    'tom_septoria': 'raw/color/Tomato___Septoria_leaf_spot/9acdb655-bca7-4223-a8e2-c201f6b3d188___Matt.S_CG 6834.JPG',
    'tom_target': 'raw/color/Tomato___Target_Spot/63a17b2f-817a-4717-9964-76ee7d6c3a63___Com.G_TgS_FL 9938.JPG',
    'tom_ylcv': 'raw/color/Tomato___Tomato_Yellow_Leaf_Curl_Virus/39b9e3b8-d3ba-4c0f-92be-e75ba728bce8___YLCV_GCREC 2711.JPG',
}

def get_image(path):
    """Скачать один снимок набора по его пути."""
    with urllib.request.urlopen(BASE + urllib.parse.quote(path), timeout=60) as r:
        return np.asarray(Image.open(io.BytesIO(r.read())).convert('RGB'))

def get_many(paths):
    """Скачать много снимков параллельно."""
    with ThreadPoolExecutor(16) as ex:
        return list(ex.map(get_image, paths))

def sample(name):
    return get_image(SAMPLES[name])

def list_dataset():
    """Имена всех снимков набора — без скачивания самих файлов (секунд 20 при первом вызове)."""
    if not os.path.exists('pv_list.txt'):
        if not os.path.exists('pv_repo'):
            subprocess.run(['git', 'clone', '-q', '--filter=blob:none', '--no-checkout', '--depth', '1',
                            'https://github.com/spMohanty/PlantVillage-Dataset', 'pv_repo'], check=True)
        out = subprocess.run(['git', '-C', 'pv_repo', 'ls-tree', '-r', '--name-only', 'HEAD', 'raw/color'],
                             capture_output=True, text=True, check=True).stdout
        with open('pv_list.txt', 'w') as f:
            f.write(out)
    names = open('pv_list.txt').read().splitlines()
    return [p for p in names if p.lower().endswith(('.jpg', '.jpeg', '.png'))]

def cls(path):
    return path.split('/')[2]

def show(images, titles=None, cols=6, size=2.3, colors=None):
    rows = (len(images) + cols - 1) // cols
    fig, axs = plt.subplots(rows, cols, figsize=(cols * size, rows * (size + 0.5)), squeeze=False)
    for ax in axs.ravel():
        ax.axis('off')
    for i, im in enumerate(images):
        ax = axs.ravel()[i]
        ax.imshow(im, cmap='gray' if im.ndim == 2 else None)
        if titles:
            ax.set_title(titles[i], fontsize=9, color=colors[i] if colors else 'black')
    plt.tight_layout(); plt.show()

print('Готово. Снимков-образцов:', len(SAMPLES))

In [ ]:
#@title Три снимка одного листа
from scipy import ndimage as ndi
from skimage import color
from PIL import ImageFilter
sharp_threshold = 1000  #@param {type:"integer"}

def sharpness(img):
    return float(ndi.laplace(color.rgb2gray(img) * 255).var())

def exposure(img):
    g = color.rgb2gray(img) * 255
    return (g < 10).mean() * 100, (g > 245).mean() * 100

leaf = sample('tom_healthy')
blur = np.asarray(Image.fromarray(leaf).filter(ImageFilter.GaussianBlur(2.5)))
k = np.zeros((15, 15)); k[7, :] = 1 / 15
motion = np.stack([ndi.convolve(leaf[..., c].astype(float), k) for c in range(3)], -1).clip(0, 255).astype(np.uint8)
ims = [leaf, blur, motion]
vals = [sharpness(im) for im in ims]
show(ims, [f'{t}\nрезкость {v:.0f} → {"принять" if v >= sharp_threshold else "отклонить"}'
           for t, v in zip(['Исходный', 'Не в фокусе', 'Смаз'], vals)], cols=3, size=3.2,
     colors=['green' if v >= sharp_threshold else 'red' for v in vals])

In [ ]:
#@title Прогоняем контроль по 60 случайным снимкам набора
ALL = list_dataset()
rng = np.random.default_rng(5)
paths = [ALL[i] for i in rng.choice(len(ALL), 60, replace=False)]
batch = get_many(paths)
sh = np.array([sharpness(im) for im in batch])
ex = np.array([exposure(im) for im in batch])
print(f'Резкость: от {sh.min():.0f} до {sh.max():.0f}; ниже порога {sharp_threshold}: {(sh < sharp_threshold).sum()} из 60')
print(f'Снимков, где больше 5 % пикселей пересвечены или провалены в чёрное: {((ex > 5).any(1)).sum()}')
order = np.argsort(sh)[:6]
show([batch[i] for i in order], [f'резкость {sh[i]:.0f}\n{cls(paths[i])[:22]}' for i in order], cols=6)

**Что сделать.** Порог 1000 подобран по одному снимку — сколько снимков набора оказалось ниже него? Посмотрите
на шесть самых «мутных»: это брак или просто мягкий фон? Порог подбирают по своим снимкам: посмотрели на границе
глазами — сдвинули.

## 4. Задание 3. Согласие на двойной разметке
Ниже — 50 снимков томата. Двое участников **независимо** размечают каждый снимок: `1` — болен, `0` — здоров.
Впишите ответы строкой из 50 цифр (пробелы можно ставить для удобства). Если поля пустые, ноутбук покажет
пример на смоделированных ответах.

In [ ]:
#@title Галерея: 50 снимков для разметки (около 30 секунд)
tom = [p for p in ALL if cls(p).startswith('Tomato')]
rng = np.random.default_rng(11)
healthy = [p for p in tom if cls(p) == 'Tomato___healthy']
sick = [p for p in tom if cls(p) != 'Tomato___healthy']
gallery = [healthy[i] for i in rng.choice(len(healthy), 25, replace=False)] + \
          [sick[i] for i in rng.choice(len(sick), 25, replace=False)]
gallery = [gallery[i] for i in rng.permutation(50)]
gal_imgs = get_many(gallery)
gal_truth = np.array([0 if cls(p) == 'Tomato___healthy' else 1 for p in gallery])
show(gal_imgs, [f'№{i + 1}' for i in range(50)], cols=10, size=1.6)

In [ ]:
#@title Считаем согласие и каппу Коэна
answers_a = ''  #@param {type:"string"}
answers_b = ''  #@param {type:"string"}
from sklearn.metrics import cohen_kappa_score

def parse(s):
    d = [int(c) for c in s if c in '01']
    return np.array(d) if len(d) == 50 else None

A_, B_ = parse(answers_a), parse(answers_b)
if A_ is None or B_ is None:
    print('ДЕМО: ответов нет (или в строке не 50 цифр) — моделируем двух разметчиков с 10 % и 25 % ошибок\n')
    r = np.random.default_rng(0)
    A_ = np.where(r.random(50) < 0.10, 1 - gal_truth, gal_truth)
    B_ = np.where(r.random(50) < 0.25, 1 - gal_truth, gal_truth)
agree = (A_ == B_).mean()
print(f'Совпали ответы: {agree:.0%}  ·  каппа Коэна: {cohen_kappa_score(A_, B_):.2f}')
print(f'Совпадение с меткой набора: A — {(A_ == gal_truth).mean():.0%}, B — {(B_ == gal_truth).mean():.0%}')
dis = np.where(A_ != B_)[0]
print('Спорные снимки:', ', '.join(f'№{i + 1}' for i in dis) or 'нет')
if len(dis):
    show([gal_imgs[i] for i in dis[:12]],
         [f'№{i + 1}: A={A_[i]} B={B_[i]}\n{cls(gallery[i]).replace("Tomato___", "")[:20]}' for i in dis[:12]], cols=6)

**Что заметить.** Каппа учитывает совпадения «по случайности»: 0 — не лучше монетки, 1 — полное согласие,
больше 0,8 — хорошо. Посмотрите на спорные снимки: это ошибки внимания или правило «болен/здоров» неясно? Второе
лечится инструкцией по разметке, а не наказанием разметчика.

In [ ]:
#@title Ловушка: высокий процент совпадений при нулевой каппе
share_healthy = 0.95  #@param {type:"slider", min:0.5, max:0.99, step:0.01}
r = np.random.default_rng(1)
t = (r.random(1000) > share_healthy).astype(int)       # 1 — болен
lazy = np.zeros(1000, int)                             # «ленивый» разметчик: всегда «здоров»
print(f'Здоровых {share_healthy:.0%}. «Всегда здоров» совпадает с истиной в {(lazy == t).mean():.0%} случаев, '
      f'каппа {cohen_kappa_score(lazy, t):.2f}')

## 5. Пятеро разметчиков: большинство против Dawid–Skene
Когда каждый снимок размечают несколько человек разного качества, голосование «большинством» считает всех равными.
Метод Dawid–Skene сам оценивает, насколько точен каждый разметчик, и взвешивает голоса — **без эталона**.

In [ ]:
#@title Моделирование: 1 000 снимков, 3 класса, 5 разметчиков
accuracies = '0.92, 0.85, 0.65, 0.55, 0.55'  #@param {type:"string"}
acc = [float(x) for x in accuracies.split(',')]
rng = np.random.default_rng(7)
N, K = 1000, 3
truth = rng.choice(K, N, p=[0.6, 0.3, 0.1])
ann = np.zeros((N, len(acc)), int)
for j, a in enumerate(acc):
    ok = rng.random(N) < a
    ann[:, j] = np.where(ok, truth, (truth + rng.integers(1, K, N)) % K)
votes = np.zeros((N, K))
for j in range(len(acc)):
    votes[np.arange(N), ann[:, j]] += 1
mv = votes.argmax(1)
T = votes / votes.sum(1, keepdims=True)
for _ in range(50):                                   # EM: оценили точность каждого → пересчитали ответы
    pi = np.full((len(acc), K, K), 1e-6)
    for j in range(len(acc)):
        for l in range(K):
            pi[j, :, l] += T[ann[:, j] == l].sum(0)
        pi[j] /= pi[j].sum(1, keepdims=True)
    prior = T.mean(0)
    logT = np.log(prior)[None, :].repeat(N, 0)
    for j in range(len(acc)):
        logT += np.log(pi[j][:, ann[:, j]].T)
    T = np.exp(logT - logT.max(1, keepdims=True)); T /= T.sum(1, keepdims=True)
ds = T.argmax(1)
print(f'Лучший разметчик в одиночку: {(ann[:, int(np.argmax(acc))] == truth).mean():.1%}')
print(f'Большинство из {len(acc)}:            {(mv == truth).mean():.1%}')
print(f'Dawid–Skene из {len(acc)}:            {(ds == truth).mean():.1%}')
print('\nТочность разметчиков: настоящая → оценённая без эталона')
for j in range(len(acc)):
    print(f'  {j + 1}: {(ann[:, j] == truth).mean():.2f} → {(pi[j].diagonal() * prior).sum():.2f}')

**Что сделать.** Сделайте всех разметчиков одинаковыми (`0.8, 0.8, 0.8, 0.8, 0.8`) — осталось ли преимущество
у Dawid–Skene? Добавьте одного очень слабого (`0.4`): кто от этого страдает сильнее — большинство или Dawid–Skene?